# Notebook 1: Knowledge Base Ingestion, Batch Embedding & ChromaDB Indexing
**Project**: Multi-Context Intelligent RAG System (Customer Support Intelligence Assistant)  
**Hardware Target**: Google Colab (Free NVIDIA T4 GPU or CPU Fallback)  
**Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, Cosine distance)  
**Persistent Storage**: Google Drive (`/content/drive/My Drive/Internship_RAG_MCP_Capstone`)

### Pipeline Overview
1. **Mount Google Drive** to access the synthetic customer support knowledge base (`03_Knowledge_Base/`).
2. **Reuse Stage 2 Ingestion & Chunker Modules** to deterministically parse and chunk all 9 documents across 4 support categories (`product_manual`, `troubleshooting`, `warranty`, `faq`).
3. **Compute Real Dense Embeddings** via `sentence-transformers/all-MiniLM-L6-v2` with L2-normalization for cosine similarity.
4. **Build ChromaDB Index** in Colab ephemeral disk (`/tmp/chroma_build/`) with scalar metadata and deterministic IDs.
5. **Execute Baseline & Category-Filtered Semantic Retrieval Tests**.
6. **Quiesce & Snapshot Complete Index** (SQLite + binary HNSW index segments + manifest) into Google Drive `05_Vector_Database_Backups/`.
7. **Perform Sandbox Restoration Test** to verify retrieval from the cloud snapshot archive with zero data corruption.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os, sys, shutil, zipfile, datetime, json, hashlib, time
from pathlib import Path

print("[1/11] Mounting Google Drive...")
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/My Drive/Internship_RAG_MCP_Capstone')
print(f"Project Cloud Root: {PROJECT_ROOT}")
assert PROJECT_ROOT.exists(), f"Google Drive project folder not found at: {PROJECT_ROOT}"

In [ ]:
# Step 2: Extract source code into Colab local workspace and configure Python path
LOCAL_REPO = Path('/content/repo')
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

# Locate latest source code backup on Google Drive
source_zip = PROJECT_ROOT / '01_Source_Code' / 'latest_source.zip'
if source_zip.exists():
    print(f"[2/11] Unpacking latest source code from {source_zip.name}...")
    with zipfile.ZipFile(source_zip, 'r') as zf:
        zf.extractall(LOCAL_REPO)
else:
    print("[2/11] latest_source.zip not found, checking 11_Backups...")
    backup_zips = sorted((PROJECT_ROOT / '11_Backups').glob('source_backup_*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    if backup_zips:
        print(f"Unpacking source code from backup: {backup_zips[0].name}...")
        with zipfile.ZipFile(backup_zips[0], 'r') as zf:
            zf.extractall(LOCAL_REPO)
    else:
        raise FileNotFoundError("No source code zip archive found on Google Drive!")

if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))

print(f"Python sys.path updated. Repository ready at {LOCAL_REPO}.")

In [ ]:
# Step 3: Install required libraries in Colab runtime
print("[3/11] Installing chromadb, sentence-transformers, and pypdf...")
!pip install -q chromadb sentence-transformers pypdf

In [ ]:
# Step 4: Import project modules and verify execution environment
import torch
import chromadb
from chromadb.config import Settings as ChromaSettings
from sentence_transformers import SentenceTransformer

from app.ingestion.loader import DocumentLoader, VALID_CATEGORIES
from app.ingestion.chunker import DocumentChunker, TextChunk
from app.ingestion.manifest import IngestionManifest

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[4/11] PyTorch compute device: {device}")
if device == "cuda":
    print(f"      GPU: {torch.cuda.get_device_name(0)}")
else:
    print("      Notice: GPU unavailable. Running small embedding model on CPU as permitted by policy.")

print(f"ChromaDB version: {chromadb.__version__}")

In [ ]:
# Step 5: Load Knowledge Base documents from Google Drive using DocumentLoader
KB_ROOT = PROJECT_ROOT / '03_Knowledge_Base'
assert KB_ROOT.exists(), f"Knowledge Base root not found: {KB_ROOT}"

loader = DocumentLoader()
chunker = DocumentChunker(chunk_size=1000, chunk_overlap=200)

print(f"[5/11] Ingesting documents from {KB_ROOT}...")
loaded_docs = []
all_chunks = []

for cat_dir in sorted(KB_ROOT.iterdir()):
    if not cat_dir.is_dir() or cat_dir.name not in VALID_CATEGORIES:
        continue
    category = cat_dir.name
    for doc_file in sorted(cat_dir.iterdir()):
        if doc_file.name == ".keep" or doc_file.is_dir():
            continue
        doc = loader.load_file(doc_file, category=category)
        loaded_docs.append(doc)
        chunks = chunker.chunk_document(doc)
        all_chunks.extend(chunks)
        print(f"  Loaded: [{category}] {doc.filename} -> {len(chunks)} chunks (Doc ID: {doc.document_id})")

print("-" * 60)
print(f"Ingestion complete: {len(loaded_docs)} documents processed, {len(all_chunks)} chunks generated.")
assert len(loaded_docs) == 9, f"Expected 9 documents, found {len(loaded_docs)}"
assert len(all_chunks) == 21, f"Expected 21 chunks, found {len(all_chunks)}"

In [ ]:
# Step 6: Load sentence-transformers/all-MiniLM-L6-v2 embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DIMENSION = 384
DISTANCE_METRIC = "cosine"

print(f"[6/11] Initializing embedding model '{MODEL_NAME}' on {device}...")
t0 = time.perf_counter()
embed_model = SentenceTransformer(MODEL_NAME, device=device)
load_time = time.perf_counter() - t0
print(f"Model loaded in {load_time:.2f}s. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")
assert embed_model.get_sentence_embedding_dimension() == DIMENSION

In [ ]:
# Step 7: Compute dense embeddings for all 21 chunks
print(f"[7/11] Computing normalized embeddings for {len(all_chunks)} chunks in batches...")
texts = [c.text for c in all_chunks]
t0 = time.perf_counter()
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)
encode_time = time.perf_counter() - t0
print(f"Encoded {len(texts)} chunks in {encode_time:.3f}s ({len(texts)/encode_time:.1f} chunks/sec).")
assert embeddings.shape == (21, 384)

In [ ]:
# Step 8: Build ChromaDB persistent index in Colab ephemeral disk
COLLECTION_NAME = "apexprint_knowledge_base"
BUILD_DIR = Path("/tmp/chroma_build")
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print(f"[8/11] Initializing ChromaDB PersistentClient at {BUILD_DIR}...")
client = chromadb.PersistentClient(
    path=str(BUILD_DIR),
    settings=ChromaSettings(anonymized_telemetry=False)
)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# Format metadata with Chroma-compatible scalar types
ids = [c.chunk_id for c in all_chunks]
metadatas = [
    {
        "document_id": c.document_id,
        "chunk_id": c.chunk_id,
        "category": c.category,
        "filename": c.metadata.get("filename", ""),
        "page_number": c.page_number if c.page_number is not None else -1,
        "source": c.source,
        "content_hash": c.content_hash,
    }
    for c in all_chunks
]

# Upsert into collection
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

count = collection.count()
print(f"ChromaDB indexing complete. Total indexed chunks: {count}")
assert count == 21, f"Expected 21 chunks in collection, found {count}"

In [ ]:
# Step 9: Run sample semantic queries (baseline + category-filtered)
print("[9/11] Running semantic retrieval verification queries...")

def test_query(q_text, category_filter=None, top_k=3):
    print("=" * 60)
    print(f"Query: '{q_text}' (Filter: {category_filter})")
    q_vec = embed_model.encode(q_text, normalize_embeddings=True).tolist()
    kwargs = {"query_embeddings": [q_vec], "n_results": top_k}
    if category_filter:
        kwargs["where"] = {"category": category_filter}
    res = collection.query(**kwargs)
    
    for i, (cid, doc_text, meta, dist) in enumerate(zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    )):
        sim = 1.0 - dist
        print(f"  Rank {i+1} | Score: {sim:.4f} (dist: {dist:.4f}) | Category: {meta['category']} | File: {meta['filename']}")
        print(f"         Text excerpt: {doc_text[:120]}...")

# 1. Wi-Fi Troubleshooting
test_query("My ApexPrint printer cannot connect to Wi-Fi. What troubleshooting instructions are available?", top_k=2)

# 2. Category-filtered Warranty Query
test_query("Does the warranty cover accidental liquid spills?", category_filter="warranty", top_k=2)

# 3. Product Manual Query
test_query("What is the maximum printing speed in pages per minute?", category_filter="product_manual", top_k=2)

In [ ]:
# Step 10: Create complete consistent snapshot and index manifest on Google Drive
print("[10/11] Packaging complete ChromaDB persistence directory...")
import sqlite3

dest_vector_dir = PROJECT_ROOT / '05_Vector_Database_Backups'
dest_vector_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_filename = f"chroma_full_snapshot_{timestamp}.zip"
manifest_filename = f"chroma_manifest_{timestamp}.json"

# Quiesce SQLite database via Online Backup API to cleanly merge WAL
quiesce_dir = Path("/tmp/chroma_quiesced")
if quiesce_dir.exists():
    shutil.rmtree(quiesce_dir)
quiesce_dir.mkdir(parents=True, exist_ok=True)

src_db = BUILD_DIR / "chroma.sqlite3"
dst_db = quiesce_dir / "chroma.sqlite3"

src_conn = sqlite3.connect(str(src_db))
dst_conn = sqlite3.connect(str(dst_db))
src_conn.backup(dst_conn)
dst_conn.close()
src_conn.close()
print("  -> SQLite WAL transactions flushed cleanly via Online Backup API.")

# Copy all segment folders, binaries, and index artifacts
for item in BUILD_DIR.iterdir():
    if item.name in ("chroma.sqlite3", "chroma.sqlite3-wal", "chroma.sqlite3-shm"):
        continue
    target = quiesce_dir / item.name
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)

# Package into temporary local zip first
local_zip = Path(f"/tmp/{zip_filename}")
with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(quiesce_dir):
        for file in files:
            p = Path(root) / file
            zf.write(p, arcname=str(p.relative_to(quiesce_dir)))

zip_bytes = local_zip.stat().st_size
zip_sha256 = hashlib.sha256(local_zip.read_bytes()).hexdigest()

# Create machine-readable manifest
manifest_data = {
    "manifest_version": "1.0",
    "collection_name": COLLECTION_NAME,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": DIMENSION,
    "distance_metric": DISTANCE_METRIC,
    "chunk_count": len(all_chunks),
    "document_ids": [d.document_id for d in loaded_docs],
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "software_versions": {
        "python": sys.version.split()[0],
        "chromadb": chromadb.__version__,
        "torch": torch.__version__,
        "compute_device": device,
    },
    "archive_filename": zip_filename,
    "archive_size_bytes": zip_bytes,
    "archive_sha256": zip_sha256
}

# Save to Google Drive
final_zip = dest_vector_dir / zip_filename
final_manifest = dest_vector_dir / manifest_filename
latest_zip = dest_vector_dir / "chroma_full_snapshot_latest.zip"
latest_manifest = dest_vector_dir / "chroma_manifest_latest.json"

shutil.copy2(local_zip, final_zip)
final_manifest.write_text(json.dumps(manifest_data, indent=2), encoding="utf-8")
shutil.copy2(local_zip, latest_zip)
latest_manifest.write_text(json.dumps(manifest_data, indent=2), encoding="utf-8")

print(f"Snapshot exported successfully to {final_zip} ({zip_bytes:,} bytes).")
print(f"Manifest written to {final_manifest}.")

In [ ]:
# Step 11: Execute full sandboxed restoration and verification test
print("[11/11] Testing restoration integrity from cloud snapshot...")
RESTORE_DIR = Path("/tmp/chroma_restored")
if RESTORE_DIR.exists():
    shutil.rmtree(RESTORE_DIR)
RESTORE_DIR.mkdir(parents=True, exist_ok=True)

# Unpack archive
with zipfile.ZipFile(latest_zip, "r") as zf:
    zf.extractall(RESTORE_DIR)

# Validate SQLite integrity
restored_db = RESTORE_DIR / "chroma.sqlite3"
assert restored_db.exists(), "Restored directory missing chroma.sqlite3!"
conn = sqlite3.connect(str(restored_db))
cur = conn.cursor()
cur.execute("PRAGMA integrity_check;")
status = cur.fetchone()[0]
conn.close()
print(f"  SQLite PRAGMA integrity_check: {status}")
assert status.lower() == "ok", f"Integrity check failed: {status}"

# Open restored collection with ChromaDB client
restored_client = chromadb.PersistentClient(
    path=str(RESTORE_DIR),
    settings=ChromaSettings(anonymized_telemetry=False)
)
restored_coll = restored_client.get_collection(COLLECTION_NAME)
restored_count = restored_coll.count()
print(f"  Restored collection chunk count: {restored_count}")
assert restored_count == 21, f"Expected 21 restored chunks, found {restored_count}"

# Run retrieval query against restored collection
q_test = "How do I reset my printer to factory defaults?"
q_vec = embed_model.encode(q_test, normalize_embeddings=True).tolist()
res = restored_coll.query(query_embeddings=[q_vec], n_results=1)
top_doc = res["documents"][0][0]
top_source = res["metadatas"][0][0]["source"]
print(f"  Post-restoration query successful!")
print(f"  Top retrieved source: {top_source}")
print(f"  Top passage preview: {top_doc[:100]}...")
print("=" * 60)
print("ALL STAGE 3 INDEXING & RESTORATION ACCEPTANCE CRITERIA VERIFIED SUCCESSFULLY!")
print("=" * 60)